In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.loader import load_cmapss_raw, add_rul

sns.set_theme(style="darkgrid")
print("Libraries loaded ✓")

## 1. Load Dataset
NASA CMAPSS FD001: 100 turbine engines, 21 sensors, simulated run-to-failure.

In [ ]:
train, test, rul = load_cmapss_raw(data_dir='../data/raw')
train = add_rul(train)

print(f"Train shape:     {train.shape}")
print(f"Test shape:      {test.shape}")
print(f"RUL series len:  {len(rul)}")
print(f"Engines in train:{train['engine_id'].nunique()}")
print(f"\nTrain columns:   {list(train.columns)}")
train.head()

## 2. Engine Lifetime Distribution

In [ ]:
max_cycles = train.groupby('engine_id')['cycle'].max()
print(f"Lifetime stats (cycles):")
print(max_cycles.describe().round(1))

plt.figure(figsize=(9, 4))
plt.hist(max_cycles, bins=20, edgecolor='black', color='steelblue')
plt.xlabel('Total Lifetime (cycles)')
plt.ylabel('Number of Engines')
plt.title('Distribution of Engine Lifetimes — FD001')
plt.tight_layout()
plt.savefig('../docs/rul_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Sensor Variance Analysis
Sensors with near-zero variance carry no information and should be dropped.

In [ ]:
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
variances = train[sensor_cols].var().sort_values()

print("Sensors by variance (ascending):")
print(variances.round(6).to_string())

LOW_VAR_THRESHOLD = 0.01
low_var = variances[variances < LOW_VAR_THRESHOLD].index.tolist()
print(f"\nSensors to DROP (var < {LOW_VAR_THRESHOLD}): {low_var}")

## 4. Sensor Degradation Patterns
Sensors that change as the engine degrades are the predictive signal.

In [ ]:
retained = [s for s in sensor_cols if s not in low_var][:9]
sample_engines = [1, 50, 100]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for i, sensor in enumerate(retained):
    ax = axes[i // 3][i % 3]
    for eng in sample_engines:
        subset = train[train['engine_id'] == eng]
        ax.plot(subset['cycle'], subset[sensor], alpha=0.8, label=f'Engine {eng}')
    ax.set_title(sensor, fontsize=10)
    ax.set_xlabel('Cycle')
axes[0][0].legend(fontsize=8)
plt.suptitle('Sensor Degradation Over Engine Lifetime', y=1.01)
plt.tight_layout()
plt.savefig('../docs/sensor_degradation.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. RUL Distribution (Capped at 125)

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(train['RUL'], bins=40, edgecolor='black', color='coral')
plt.axvline(125, color='red', linestyle='--', label='Cap = 125 cycles')
plt.xlabel('RUL (cycles)')
plt.ylabel('Count')
plt.title('RUL Distribution After 125-cycle Cap')
plt.legend()
plt.tight_layout()
plt.show()

print(f"RUL range: [{train['RUL'].min()}, {train['RUL'].max()}]")
print(f"Mean RUL: {train['RUL'].mean():.1f} cycles")

## Key Findings
- Sensors to drop (near-zero variance): sensor_1, sensor_5, sensor_6, sensor_10, sensor_16, sensor_18, sensor_19
- Retained sensors show clear degradation trends (especially sensor_2, sensor_4, sensor_11)
- RUL cap at 125 cycles prevents label noise in early healthy cycles